# Heathrow SQL Analysis and Dashboard Exports
## 04 — SQLite Reporting and Tableau Preparation

### Purpose
Store the prepared data in SQLite, use SQL to analyse trends,
variances and scenarios, and export reporting tables for Tableau.

### Reporting principles
- Distinguish reported passenger actuals from passenger forecasts.
- Label monthly financial figures as modelled estimates.
- Keep the 2025 comparison baseline separate from forecast scenarios.
- Reconcile database totals to the prepared source tables.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATABASE_DIR = PROJECT_ROOT / "database"
DATABASE_DIR.mkdir(exist_ok=True)

DATABASE_PATH = DATABASE_DIR / "heathrow_model.sqlite"

print("Database path:", DATABASE_PATH)

Database path: /Users/dafyddjones/projects/heathrow-forecasting-model/database/heathrow_model.sqlite


In [2]:
financial_model = pd.read_csv(
    PROCESSED_DIR / "heathrow_financial_model_2026.csv",
    parse_dates=["month"]
)

print("Rows:", len(financial_model))
print("Columns:", len(financial_model.columns))

financial_model[
    ["month", "passengers", "financial_data_status"]
].head()

Rows: 12
Columns: 23


,month,passengers,financial_data_status
0,2026-01-01,6460877,modelled_estimate
1,2026-02-01,5819080,modelled_estimate
2,2026-03-01,6645184,modelled_estimate
3,2026-04-01,6721032,modelled_estimate
4,2026-05-01,7121212,modelled_estimate


In [3]:
financial_model_sql = financial_model.copy()
financial_model_sql["month"] = (
    financial_model_sql["month"].dt.strftime("%Y-%m-%d")
)

with sqlite3.connect(DATABASE_PATH) as connection:
    financial_model_sql.to_sql(
        "financial_model_monthly",
        connection,
        if_exists="replace",
        index=False
    )

print("Saved SQLite table: financial_model_monthly")

Saved SQLite table: financial_model_monthly


In [4]:
query = """
SELECT
    COUNT(*) AS rows_in_database,
    COUNT(DISTINCT month) AS unique_months,
    MIN(month) AS first_month,
    MAX(month) AS last_month,
    SUM(passengers) AS full_year_passengers
FROM financial_model_monthly;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    database_check = pd.read_sql_query(query, connection)

database_check

,rows_in_database,unique_months,first_month,last_month,full_year_passengers
0,12,12,2026-01-01,2026-12-01,84088953


In [5]:
query = """
SELECT
    SUM(modelled_revenue_gbp_m) AS revenue_gbp_m,
    SUM(modelled_operating_cost_gbp_m) AS operating_cost_gbp_m,
    SUM(modelled_adjusted_ebitda_gbp_m) AS adjusted_ebitda_gbp_m,
    100.0 * SUM(modelled_adjusted_ebitda_gbp_m)
        / SUM(modelled_revenue_gbp_m) AS adjusted_ebitda_margin_pct
FROM financial_model_monthly;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    sql_financial_summary = pd.read_sql_query(query, connection)

sql_financial_summary.round(2)

,revenue_gbp_m,operating_cost_gbp_m,adjusted_ebitda_gbp_m,adjusted_ebitda_margin_pct
0,3606.95,1586.89,2020.06,56.0


In [6]:
query = """
SELECT
    SUM(revenue_variance_gbp_m) AS revenue_variance_gbp_m,
    SUM(cost_variance_gbp_m) AS cost_variance_gbp_m,
    SUM(ebitda_variance_gbp_m) AS ebitda_variance_gbp_m,
    100.0 * SUM(ebitda_variance_gbp_m)
        / SUM(baseline_adjusted_ebitda_gbp_m)
        AS ebitda_variance_pct
FROM financial_model_monthly
WHERE month <= '2026-09-01';
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    sql_ytd_variances = pd.read_sql_query(query, connection)

sql_ytd_variances.round(2)

,revenue_variance_gbp_m,cost_variance_gbp_m,ebitda_variance_gbp_m,ebitda_variance_pct
0,-16.05,-2.11,-13.94,-0.92


In [7]:
scenarios_monthly = pd.read_csv(
    PROCESSED_DIR / "heathrow_scenarios_monthly_2026.csv"
)

with sqlite3.connect(DATABASE_PATH) as connection:
    scenarios_monthly.to_sql(
        "scenarios_monthly",
        connection,
        if_exists="replace",
        index=False
    )

print("Scenario rows loaded:", len(scenarios_monthly))

Scenario rows loaded: 36


In [8]:
query = """
SELECT
    scenario,
    SUM(scenario_passengers) AS passengers,
    SUM(modelled_revenue_gbp_m) AS revenue_gbp_m,
    SUM(modelled_operating_cost_gbp_m) AS operating_cost_gbp_m,
    SUM(modelled_adjusted_ebitda_gbp_m) AS adjusted_ebitda_gbp_m,
    100.0 * SUM(modelled_adjusted_ebitda_gbp_m)
        / SUM(modelled_revenue_gbp_m) AS adjusted_ebitda_margin_pct
FROM scenarios_monthly
GROUP BY scenario
ORDER BY adjusted_ebitda_gbp_m DESC;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    sql_scenario_summary = pd.read_sql_query(query, connection)

sql_scenario_summary.round(2)

,scenario,passengers,revenue_gbp_m,operating_cost_gbp_m,adjusted_ebitda_gbp_m,adjusted_ebitda_margin_pct
0,Upside,85148375.45,3652.40,1592.87,2059.53,56.39
1,Baseline,84088953.00,3606.95,1586.89,2020.06,56.00
2,Downside,83029530.55,3561.51,1580.91,1980.60,55.61


In [9]:
SQL_DIR = PROJECT_ROOT / "sql"
SQL_DIR.mkdir(exist_ok=True)

query_path = SQL_DIR / "scenario_summary.sql"
query_path.write_text(query.strip() + "\n", encoding="utf-8")

print("Saved:", query_path.name)

Saved: scenario_summary.sql


In [10]:
TABLEAU_DIR = PROJECT_ROOT / "tableau"
TABLEAU_DIR.mkdir(exist_ok=True)

sql_scenario_summary.to_csv(
    TABLEAU_DIR / "heathrow_scenario_summary_2026.csv",
    index=False
)

print("Exported: heathrow_scenario_summary_2026.csv")

Exported: heathrow_scenario_summary_2026.csv


In [11]:
monthly_dashboard_query = """
SELECT
    month,
    passengers,
    passenger_status,
    baseline_passengers,
    modelled_revenue_gbp_m,
    baseline_revenue_gbp_m,
    revenue_variance_gbp_m,
    revenue_variance_pct,
    modelled_operating_cost_gbp_m,
    baseline_operating_cost_gbp_m,
    cost_variance_gbp_m,
    modelled_adjusted_ebitda_gbp_m,
    baseline_adjusted_ebitda_gbp_m,
    ebitda_variance_gbp_m,
    ebitda_variance_status,
    financial_data_status
FROM financial_model_monthly
ORDER BY month;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    monthly_dashboard = pd.read_sql_query(
        monthly_dashboard_query, connection
    )

print("Dashboard rows:", len(monthly_dashboard))
monthly_dashboard.head()

Dashboard rows: 12


,month,passengers,passenger_status,baseline_passengers,modelled_revenue_gbp_m,baseline_revenue_gbp_m,revenue_variance_gbp_m,revenue_variance_pct,modelled_operating_cost_gbp_m,baseline_operating_cost_gbp_m,cost_variance_gbp_m,modelled_adjusted_ebitda_gbp_m,baseline_adjusted_ebitda_gbp_m,ebitda_variance_gbp_m,ebitda_variance_status,financial_data_status
0,2026-01-01,6460877,reported_actual,6319076,277.136030,271.053548,6.082482,2.244015,129.156129,128.355820,0.800309,147.979901,142.697728,5.282173,Favourable,modelled_estimate
1,2026-02-01,5819080,reported_actual,5709458,249.606474,244.904294,4.702180,1.920007,125.533899,124.915204,0.618694,124.072575,119.989090,4.083486,Favourable,modelled_estimate
2,2026-03-01,6645184,reported_actual,6218374,285.041785,266.733987,18.307798,6.863691,130.196336,127.787469,2.408868,154.845448,138.946518,15.898930,Favourable,modelled_estimate
3,2026-04-01,6721032,reported_actual,7098983,288.295246,304.507262,-16.212016,-5.324016,130.624414,132.757527,-2.133113,157.670832,171.749735,-14.078903,Unfavourable,modelled_estimate
4,2026-05-01,7121212,reported_actual,7204388,305.460763,309.028555,-3.567792,-1.154519,132.882985,133.352421,-0.469436,172.577778,175.676135,-3.098356,Unfavourable,modelled_estimate


In [12]:
print("Total missing values:", monthly_dashboard.isna().sum().sum())
print("First month:", monthly_dashboard["month"].min())
print("Last month:", monthly_dashboard["month"].max())

Total missing values: 0
First month: 2026-01-01
Last month: 2026-12-01


In [13]:
monthly_dashboard.to_csv(
    TABLEAU_DIR / "heathrow_monthly_dashboard_2026.csv",
    index=False
)

(SQL_DIR / "monthly_dashboard.sql").write_text(
    monthly_dashboard_query.strip() + "\n",
    encoding="utf-8"
)

print("Exported: heathrow_monthly_dashboard_2026.csv")
print("Saved: monthly_dashboard.sql")

Exported: heathrow_monthly_dashboard_2026.csv
Saved: monthly_dashboard.sql


### SQLite checks and Tableau exports

- Monthly financial model: 12 unique months, January–December 2026.
- Full-year passenger outlook: 84,088,953.
- SQL financial totals reconcile to the Python model.
- SQL January–September variances reconcile to the Python results.
- Scenario table: 36 rows, covering 12 months and three scenarios.
- SQL annual scenario results reconcile to the Python summaries.
- Monthly dashboard table contains no missing values.

### Tableau inputs

- tableau/heathrow_monthly_dashboard_2026.csv
- tableau/heathrow_scenario_summary_2026.csv

Reusable queries are saved in the sql folder.
All monthly financial figures are modelled estimates.